# A2-Lite — fewer layers, same MonoDETR family
Latest revision: **A2-LITE-E1-D1-ONNX-CPU-SCREEN-2026-10-12-r1**

Load the original A2 epoch130 checkpoint strictly. Keep the MobileNetV4 Medium backbone, depth map/depth encoder, 256-channel features,50 object queries,4 feature levels,384×1280 image and native3D output heads. Retain visual encoder layer0 and detection decoder layer0, with the matching stage0 prediction heads. Remove layers1/2. This is a layer-pruned initialization, NOT a recovered/accuracy-qualified detector. No training or distillation occurs.

Export ordinary fixed-shape ONNX, verify all16 fixtures against the candidate's CPU PyTorch outputs, and compare host CPU timing with original A2. Host timing is NOT iPhone timing. Do not claim A2-level accuracy from inherited weights. We must measure on the actual iPhone before choosing recovery training. The phone package must be added as a separate A2_LITE_ONNX resource in the existing benchmark app, never replace original A2 assets.

Run sections1–5 in order. CPU runtime is sufficient; a GPU runtime is also fine but unused. No CUDA extension build, old virtual environment, KITTI dataset restore, Torch replacement or Core ML conversion is needed. The reviewed portable tensor patch has Core ML in its filename, but no Core ML library is installed or called. Use a new RUN_ID after a partial/changed export; preserve previous outputs. Send a2_lite_screen_results.zip. Keep the derived initialization checkpoint on Drive for possible future training.

## 1. Mount Drive, paths and every helper
The two input folders/records already come from completed A2/M68 work. A downloaded M68 export ZIP also works for BASELINE_EXPORT. No cells from an older notebook are needed.

In [ ]:
# A2-LITE-E1-D1-ONNX-CPU-SCREEN-2026-10-12-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys
REVISION = 'A2-LITE-E1-D1-ONNX-CPU-SCREEN-2026-10-12-r1'
RUN_ID = 'a2_lite_e1_d1_cpu_screen_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID): raise ValueError('Use lowercase letters, digits and underscores in RUN_ID')
MOBILE_REPO = Path('/content/mobile_adas3d')
A2_RECORD = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_m66_r0_a2_feature/m66_r0_a2_vehicle_feature_r1/m66_manifest.json')
BASELINE_EXPORT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_m68_onnx_cpu/m68_a2_onnx_cpu_r3/export')
CHECKPOINT_OVERRIDE = None  # Set Path(...) only if the same A2 epoch130 file moved; SHA256 is still enforced.
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite')/RUN_ID
SOURCE_REPO = Path('/content/MonoDETR_A2LITE_'+RUN_ID)
PYTHON = Path(sys.executable)
SCRIPT = MOBILE_REPO/'scripts/a2_lite_onnx.py'
LOGS = OUTPUT/'colab_logs'; LOGS.mkdir(parents=True,exist_ok=True)
READY = False
def run_logged(command,name,cwd=None):
    command=list(map(str,command))
    path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True)
    tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n'); log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally: process.stdout.close()
        code=process.wait(); log.write(f'\nEXIT CODE: {code}\n'); log.flush()
    if code: raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY: raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-u',SCRIPT,name,'--output',OUTPUT,*extra],name,MOBILE_REPO)
print(REVISION,'\nOutput:',OUTPUT,'\nNo training; actual iPhone timing comes next.')


## 2. Update clean main and install export dependencies
Torch/CUDA are not installed or replaced. ONNX Runtime1.30.0 matches the existing phone dependency.

In [ ]:
READY=False
if not MOBILE_REPO.exists(): run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty: raise RuntimeError('Preserve tracked edits; use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main': raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file(): raise FileNotFoundError(SCRIPT)
for path in (A2_RECORD,BASELINE_EXPORT):
    if not path.exists(): raise FileNotFoundError(f'{path}: correct the path in section1')
run_logged([PYTHON,'-m','pip','install','-q','timm==1.0.27','numpy==2.1.3','onnx==1.20.1','onnxruntime==1.30.0','pyyaml','scipy','pillow','opencv-python-headless'],'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnx,onnxruntime as ort; assert timm.__version__=='1.0.27'; assert ort.__version__=='1.30.0'; print(torch.__version__,timm.__version__,onnx.__version__,ort.__version__)"],'versions')
READY=True


## 3. Tests and original checkpoint/source verification
The record supplies the frozen A2 checkpoint path and exact model configuration, not an old runtime. A dedicated pinned checkout receives portable inference patches only. All16 original saved-input CPU fixtures must pass.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_onnx.py','-v'],'tests',MOBILE_REPO)
extra=['--checkpoint',CHECKPOINT_OVERRIDE] if CHECKPOINT_OVERRIDE is not None else []
stage('prepare','--a2-record',A2_RECORD,'--teacher-export',BASELINE_EXPORT,'--repo',SOURCE_REPO,*extra)


## 4. Layer pruning, ONNX export and paired host timing
First verify the reconstructed full A2's decoded outputs. Then keep layer0 and stage0 heads, with refinement aliases rebound correctly. The depth module is unchanged. Verify candidate ONNX versus candidate PyTorch on16 fixed inputs. Timing is host CPU/model-only, fixed original-then-lite order; do not use it as an iPhone performance gate.

In [ ]:
stage('export')
REPORT=json.loads((OUTPUT/'export/a2_lite_export_report.json').read_text())
assert REPORT['complete'] and not REPORT['training_performed'] and REPORT['optimizer_steps']==0
print(json.dumps({k:REPORT[k] for k in ('pruning','host_cpu_original','host_cpu_lite','host_cpu_p50_speedup')},indent=2))
print('Send the ZIP next. No accuracy qualification and no phone result yet.')


## 5. Download the screen bundle
Also run this after an export failure so the logs are included. The ZIP includes ONNX and fixed-input fixtures when successful, but excludes original teacher assets and checkpoints. Do not connect the phone yet: first return this bundle so its hashes can be pinned in the existing benchmark app.

In [ ]:
stage('bundle')
BUNDLE=OUTPUT/'a2_lite_screen_results.zip'
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
